# 04 — Base de datos PostgreSQL: esquema relacional, carga y consultas

**Objetivo:** pasar de un archivo plano a una **base de datos relacional** con reglas de integridad, índices, vistas para Power BI y un usuario de solo lectura.

**Requisitos:** PostgreSQL instalado y el `.env` con `PG_PASSWORD`, `PG_BI_USER` y `PG_BI_PASSWORD`.

**Resultados que guarda:** base `cobranza` con 9 tablas y 2 vistas; `docs/validacion_carga.csv`.

## 1. Preparación

In [1]:
import sys
from pathlib import Path

RAIZ = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

import pandas as pd
from sqlalchemy import text

from src.config import RUTAS
from src.objetivo import asignar_conjunto
from src.bd import (crear_base_si_no_existe, obtener_motor, ejecutar_sql, crear_rol_lectura,
                    preparar_tablas, cargar_tablas, validar_carga)

## 2. Crear la base `cobranza`
Se conecta a la base de sistema `postgres` y crea `cobranza` solo si no existe.

In [2]:
crear_base_si_no_existe()
motor = obtener_motor()
with motor.connect() as con:
    print(con.execute(text('SELECT version()')).scalar())

La base 'cobranza' ya existe.
PostgreSQL 17.11 on x86_64-windows, compiled by msvc-19.44.35229, 64-bit


## 3. Crear el esquema (tablas, llaves y restricciones)
El script borra y recrea las tablas: así el notebook se puede ejecutar las veces que haga falta (**idempotente**).

In [3]:
ejecutar_sql(motor, '01_esquema.sql')

✅ Ejecutado sql/01_esquema.sql


## 4. Preparar los datos
Agregamos la etiqueta de **conjunto** (entrenamiento, validación, inmadura o excluida) y separamos la población en las 4 tablas de datos.

In [4]:
pob = pd.read_parquet(RUTAS['data_interim'] / 'poblacion_objetivo.parquet')
pob['conjunto'] = asignar_conjunto(pob)
print(pob['conjunto'].value_counts().to_string())

tablas = preparar_tablas(pob)
for nombre, df in tablas.items():
    print(f'{nombre:25s} {df.shape[0]:>7,} filas x {df.shape[1]:>2} columnas')

conjunto
entrenamiento    8543
inmadura         5196
validacion       3163
excluida          108


deudores                   15,811 filas x  4 columnas
creditos                   17,010 filas x 29 columnas
perfil_solicitud           17,010 filas x 42 columnas
resultados_recuperacion    17,010 filas x 22 columnas


## 5. Cargar a PostgreSQL
Todo en **una sola transacción**: si algo falla, no queda una carga a medias.

In [5]:
cargar_tablas(motor, tablas)

  deudores                   15,811 filas


  creditos                   17,010 filas


  perfil_solicitud           17,010 filas


  resultados_recuperacion    17,010 filas
✅ Carga completa


## 6. Índices, vistas y permisos

In [6]:
ejecutar_sql(motor, '02_indices.sql')
ejecutar_sql(motor, '03_vistas.sql')
crear_rol_lectura(motor)
ejecutar_sql(motor, '04_permisos.sql')

✅ Ejecutado sql/02_indices.sql
✅ Ejecutado sql/03_vistas.sql
✅ Rol de solo lectura 'lector_bi' actualizado.
✅ Ejecutado sql/04_permisos.sql


## 7. Validar la carga
Comparamos conteos y sumas de control entre pandas y PostgreSQL. Si no coinciden, algo se perdió en la carga.

📋 **Pega esta tabla.**

In [7]:
validacion = validar_carga(motor, tablas)
validacion.to_csv(RUTAS['docs'] / 'validacion_carga.csv', index=False, encoding='utf-8-sig')
validacion

✅ Validación superada


,tabla,control,pandas,postgres,ok
0,deudores,COUNT(*),1.581100e+04,1.581100e+04,True
1,creditos,COUNT(*),1.701000e+04,1.701000e+04,True
2,perfil_solicitud,COUNT(*),1.701000e+04,1.701000e+04,True
3,resultados_recuperacion,COUNT(*),1.701000e+04,1.701000e+04,True
4,creditos,SUM(monto_original),1.092866e+08,1.092866e+08,True
5,resultados_recuperacion,SUM(exposicion_castigo),7.980675e+07,7.980675e+07,True
6,resultados_recuperacion,SUM(recuperacion_total),5.103981e+06,5.103981e+06,True
7,resultados_recuperacion,SUM(objetivo_a),3.683000e+03,3.683000e+03,True


## 8. Probar que la integridad funciona
Intentamos insertar datos **inválidos** a propósito. La base debe **rechazarlos**: esa es la evidencia de que las reglas de calidad viven en la base y no solo en el código.

In [8]:
pruebas = {
    'Crédito con deudor inexistente (llave foránea)':
        "INSERT INTO creditos (credito_id, loan_key, listing_key, deudor_id, fecha_originacion, plazo_meses, monto_original, estado_credito) VALUES (999999, 'X', 'X', 999999, '2010-01-01', 36, 1000, 'Chargedoff')",
    'Objetivo A distinto de 0 o 1 (CHECK)':
        'UPDATE resultados_recuperacion SET objetivo_a = 5 WHERE credito_id = 1',
    'Tasa de recuperación mayor a 100 % (CHECK)':
        'UPDATE resultados_recuperacion SET objetivo_b_tasa = 1.5 WHERE credito_id = 1',
    'Crédito duplicado (UNIQUE)':
        "INSERT INTO creditos SELECT * FROM creditos WHERE credito_id = 1",
}
for nombre, sql in pruebas.items():
    try:
        with motor.begin() as con:
            con.exec_driver_sql(sql)
        print(f'❌ {nombre}: la base lo aceptó')
    except Exception as e:
        print(f'✅ {nombre}: rechazado ({type(e.orig).__name__})')

✅ Crédito con deudor inexistente (llave foránea): rechazado (ForeignKeyViolation)
✅ Objetivo A distinto de 0 o 1 (CHECK): rechazado (CheckViolation)
✅ Tasa de recuperación mayor a 100 % (CHECK): rechazado (CheckViolation)
✅ Crédito duplicado (UNIQUE): rechazado (UniqueViolation)


## 9. Probar el usuario de solo lectura
Power BI y la aplicación se conectan con `lector_bi`. Debe poder leer las **vistas**, pero no las tablas con datos personales ni modificar nada.

In [9]:
motor_bi = obtener_motor(rol='lector')
with motor_bi.connect() as con:
    n = con.execute(text('SELECT COUNT(*) FROM vw_cartera_cobranza')).scalar()
    print(f'✅ lector_bi lee vw_cartera_cobranza: {n:,} filas')

for sql in ['SELECT * FROM perfil_solicitud LIMIT 1', 'DELETE FROM segmentos']:
    try:
        with motor_bi.begin() as con:
            con.exec_driver_sql(sql)
        print(f'❌ lector_bi pudo ejecutar: {sql}')
    except Exception as e:
        print(f'✅ Bloqueado para lector_bi: {sql}')
motor_bi.dispose()

✅ lector_bi lee vw_cartera_cobranza: 17,010 filas


✅ Bloqueado para lector_bi: SELECT * FROM perfil_solicitud LIMIT 1
✅ Bloqueado para lector_bi: DELETE FROM segmentos


## 10. Consultas de negocio (`sql/05_consultas_ejemplo.sql`)
📋 **Pega las salidas de Q1, Q2, Q3 y Q5.**

In [10]:
consultas = (RUTAS['sql'] / '05_consultas_ejemplo.sql').read_text(encoding='utf-8')
bloques = [b.strip() for b in consultas.split('-- Q')[1:]]

with motor.connect() as con:
    for bloque in bloques:
        titulo, sql = bloque.split('\n', 1)
        print(f'\n===== Q{titulo} =====')
        print(pd.read_sql(text(sql.strip().rstrip(';')), con).to_string(index=False))


===== Q1. Línea base: ¿cuánto se recupera hoy sin modelo? =====
 creditos  exposicion_usd  recuperado_usd  pct_recupera_algo  tasa_recuperacion_pct
    17010     79806754.92      5103981.04              21.65                    6.4

===== Q2. Tamaño de cada conjunto (partición temporal y cartera inmadura) =====
     conjunto  creditos  pct_recupera
entrenamiento      8543          23.5
     excluida       108           0.9
     inmadura      5196          14.1
   validacion      3163          29.7

===== Q3. Deudores con más de un crédito en cobranza (justifica la tabla deudores) =====
 creditos_en_cobranza  deudores
                    1     14612
                    2      1199

===== Q4. Recuperación por calificación, solo créditos maduros =====
calificacion  creditos  pct_recupera  tasa_cartera_pct
           A       888          25.6               6.8
          AA       571          26.1               6.1
           B      1457          26.7               8.1
           C      21

  tramo_default  creditos  pct_recupera
  01: cuota 0-5       754          64.3
 02: cuota 6-11      3404          16.6
03: cuota 12-23      4754          23.2
  04: cuota 24+      2789          28.4
       sin dato         5          80.0

===== Q6. Cosechas: resumen por año de cierre (vista para Power BI) =====
 anio_cierre estado_credito  creditos  exposicion_usd  recuperado_usd  pct_recupera_algo  tasa_recuperacion_pct  creditos_maduros
        2006     Chargedoff         3        12612.69        13837.81             100.00                 109.71                 3
        2006      Defaulted        97       409955.39         4780.66               9.28                   1.17                97
        2007     Chargedoff       224      1005927.93        71765.54              33.48                   7.13               224
        2007      Defaulted      1458      6330855.33        80201.44              12.55                   1.27              1458
        2008     Chargedoff      21

## 11. Inventario final de la base

In [11]:
with motor.connect() as con:
    objetos = pd.read_sql(text("""
        SELECT table_name AS objeto,
               CASE table_type WHEN 'VIEW' THEN 'vista' ELSE 'tabla' END AS tipo
        FROM information_schema.tables
        WHERE table_schema = 'public'
        ORDER BY tipo, objeto"""), con)
    objetos['filas'] = [con.execute(text(f'SELECT COUNT(*) FROM {o}')).scalar() for o in objetos['objeto']]
objetos

,objeto,tipo,filas
0,campana_asignaciones,tabla,0
1,campanas,tabla,0
2,canales,tabla,0
3,creditos,tabla,17010
4,deudores,tabla,15811
5,perfil_solicitud,tabla,17010
6,predicciones,tabla,0
7,resultados_recuperacion,tabla,17010
8,segmentos,tabla,0
9,vw_cartera_cobranza,vista,17010
